In [1]:
!git clone https://github.com/abolfazlaghdaee/Irony_Detection

Cloning into 'Irony_Detection'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 125 (delta 19), reused 26 (delta 6), pack-reused 77 (from 1)
Receiving objects: 100% (125/125), 83.58 MiB | 17.75 MiB/s, done.
Resolving deltas: 100% (55/55), done.


In [2]:
%cd Irony_Detection

/content/Irony_Detection


In [3]:
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import load_dataset, Dataset



In [4]:
df = pd.read_csv("data/Preprocessed_data.csv")
df.head()

,tweet_with_emoji_meaning,label
0,پیرمرد وصیت احدی تاکید احدی درد دل نکنید بعدش ...,0
1,مجوز بده ملت ماشین استاندارد بتونن بیارن سوار...,0
2,دیت دختره دید زشتم می‌خواست پاشه بره آینهی دست...,1
3,اکیپ دخترونه هست پایه قراراست کلاس نمیذاره زود...,0
4,۵۰۰ نفری مراسم سالگرد پدر همسرم شرکت فوتی‌ی ۴۰...,0


In [5]:
dataset = Dataset.from_pandas(df)

In [6]:
model_name = "HooshvareLab/bert-base-parsbert-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [7]:
def tokenize(batch):
    return tokenizer(batch['tweet_with_emoji_meaning'], padding="max_length", truncation=True, max_length=128)


dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/14676 [00:00<?, ? examples/s]

In [8]:
dataset = dataset.train_test_split(test_size=0.2)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [9]:
num_labels = len(set(df['label']))
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)


pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/654M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at HooshvareLab/bert-base-parsbert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


In [11]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-12-2388959174.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [13]:
trainer.train()


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abolfzl (abolfzl-university-of-kashan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.469200,0.464695,0.802793,0.802868,0.802793,0.802674
2,0.357300,0.658657,0.773842,0.789695,0.773842,0.769574


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.469200,0.464695,0.802793,0.802868,0.802793,0.802674
2,0.357300,0.658657,0.773842,0.789695,0.773842,0.769574
3,0.255700,0.927314,0.805518,0.805669,0.805518,0.805368
4,0.137400,1.111480,0.799728,0.799796,0.799728,0.799748


TrainOutput(global_step=5872, training_loss=0.3035386972921096, metrics={'train_runtime': 1848.9286, 'train_samples_per_second': 25.398, 'train_steps_per_second': 3.176, 'total_flos': 3088923789926400.0, 'train_loss': 0.3035386972921096, 'epoch': 4.0})